# Collecting rollouts with OpenEnv for supervised training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meta-pytorch/OpenEnv/blob/main/examples/sft_warmup.ipynb)

OpenEnv environments are not only useful for RL training — they are also a natural tool for **collecting rollouts that become supervised training data**. The environment handles episode management, automatic scoring, and reproducibility, so you get a reward-labeled dataset without writing any of that infrastructure yourself.

**What you'll build:**
- ~300 rollouts collected from `gpt-5-mini` running inside `reasoning_gym_env`, scored automatically by the environment
- A filtered, inspected SFT dataset with token-length analysis
- A fine-tuned `Qwen/Qwen3-1.7B` checkpoint trained on the collected rollouts
- A before/after eval table showing the improvement from training on environment-collected data

**Next step:** use the checkpoint as a warm-start for GRPO in the [End-to-end walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough).

---

## 1. Install dependencies

In [ ]:
!pip install -q openai trl openenv
!pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym
!pip install -Uq "transformers>=5.3.0"

---

## 2. Set your credentials

In [ ]:
import getpass, os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

YOUR_HF_USERNAME = "your-username"  # replace with your Hugging Face username
assert YOUR_HF_USERNAME != "your-username", "Replace YOUR_HF_USERNAME with your Hugging Face username"

---

## 3. Define the system prompt

Use the same prompt as the GRPO tutorial so the SFT checkpoint is a drop-in when you continue with GRPO.

In [ ]:
SYSTEM_PROMPT = """You are a careful arithmetic assistant.

You will be given a chain of integer additions. Compute the result and submit it as a single number.

Rules:
1. Read the question carefully.
2. Use the tool `answer` exactly once with your final number.
3. The answer must be a single integer with no units or explanation.
"""

---

## 4. Configure data collection

`DATASET_CONFIG` controls the difficulty of the `chain_sum` problems the environment generates:
`min_terms`/`max_terms` set how many integers are added together, and `min_digits`/`max_digits` set
how many digits each integer has. At these settings each problem is a sum of 2–3 two-digit numbers —
easy enough for `gpt-5-mini` to answer correctly ~90% of the time, which gives a clean training signal after filtering.

`N_EPISODES` is the number of problems to collect. 300 is enough to get ~270 correct examples after filtering, which is sufficient for format compliance training.

In [ ]:
DATASET_CONFIG = {
    "min_terms": 2,
    "max_terms": 3,
    "min_digits": 2,
    "max_digits": 2,
}

N_EPISODES = 300

---

## 5. Collect rollouts

The `openenv collect` pipeline runs the teacher model inside the environment and records every episode.
The environment's `step()` reward is written alongside the messages, so filtering by correctness requires
no additional scoring code.

The same pipeline is available as a Python API — the CLI wraps these exact calls:

> **CLI equivalent:** `openenv collect reasoning_gym:chain_sum --provider openai --model gpt-5-mini --num-episodes 300 --push-to-hub <user>/chain-sum-rollouts ...`

In [ ]:
import os
from openenv.core.harness import HarnessRunLimits, MCPHarnessAdapter
from openenv.core.harness.collect import build_model_step, CollectRunner, RolloutSerializer
from openenv.core.llm_client import create_llm_client
from reasoning_gym_env.client import ReasoningGymEnv
from reasoning_gym_env.harness import ReasoningGymSessionFactory

client = create_llm_client(
    provider="openai",
    model="gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=1024,
)
model_step = build_model_step(client, system_prompt=SYSTEM_PROMPT)

factory = ReasoningGymSessionFactory(
    lambda: ReasoningGymEnv(base_url="https://sergiopaniego-reasoning-gym.hf.space"),
    dataset_name="chain_sum",
    dataset_config=DATASET_CONFIG,
)

serializer = RolloutSerializer("./rollouts")
runner = CollectRunner(
    session_factory=factory,
    harness_adapter=MCPHarnessAdapter(),
    serializer=serializer,
    limits=HarnessRunLimits(max_turns=9),
)

result = runner.run(model_step=model_step, num_episodes=N_EPISODES)
print(f"Collected={result.num_collected} dropped={result.num_dropped} avg_reward={result.avg_reward:.3f} success_rate={result.success_rate:.0%}")

In [ ]:
from openenv.core.harness.collect import push_to_hf_hub

url = push_to_hf_hub(
    output_dir="./rollouts",
    repo_id=f"{YOUR_HF_USERNAME}/chain-sum-rollouts",
)
print(f"Dataset at: {url}")

The collected `messages` use the standard OpenAI `tool_calls` format. Convert to Qwen3's
`<tool_call>` text format — GRPOTrainer produces this same format during RL, making the
SFT checkpoint a direct drop-in.

In [ ]:
from datasets import load_dataset
import json

ds = load_dataset(f"{YOUR_HF_USERNAME}/chain-sum-rollouts", split="train")
raw_rollouts = list(ds)
print(f"Collected {len(raw_rollouts)} episodes")

def to_qwen3_messages(record):
    converted = []
    for msg in record["messages"]:
        if msg["role"] == "tool":
            continue  # strip env responses; SFT only needs the assistant turn
        if msg["role"] == "assistant" and msg.get("tool_calls"):
            tc = msg["tool_calls"][0]
            args = json.loads(tc["function"]["arguments"])
            answer_str = args.get("answer", "")
            tool_call_text = (
                "<tool_call>\n"
                + json.dumps({"name": "answer", "arguments": {"answer": answer_str}})
                + "\n</tool_call>"
            )
            converted.append({"role": "assistant", "content": tool_call_text})
        else:
            converted.append(msg)
    return {"messages": converted, "reward": record["reward"]}

rollouts = [to_qwen3_messages(r) for r in raw_rollouts]

---

## 6. Filter the dataset

Keep only episodes where the teacher answered correctly. The environment's reward signal does the
labelling — no manual annotation needed.

In [ ]:
correct = [r for r in rollouts if r["reward"] == 1.0]
print(f"Correct: {len(correct)} / {len(rollouts)} ({len(correct)/len(rollouts):.1%})")

---

## 7. Inspect the dataset before training

Always look at your data before training. Automated collection can introduce unexpected patterns that the
student model will learn to imitate.

In [ ]:
import random

for row in random.sample(correct, 3):
    question = row["messages"][0]["content"]
    response = row["messages"][1]["content"]
    print(f"Q: {question}")
    print(f"A: {response}")
    print()

---

## 8. Measure token lengths

Set `max_seq_length` to the 99th percentile so fewer than 1% of examples are truncated.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

lengths = []
for row in correct:
    text = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )
    ids = tokenizer.encode(text)
    lengths.append(len(ids))

lengths = np.array(lengths)
MAX_SEQ_LEN = int(np.percentile(lengths, 99)) + 16

print(
    f"p50={np.percentile(lengths, 50):.0f}  "
    f"p95={np.percentile(lengths, 95):.0f}  "
    f"p99={np.percentile(lengths, 99):.0f}  "
    f"max={lengths.max()}"
)
print(f"Setting MAX_SEQ_LEN = {MAX_SEQ_LEN}")

---

## 9. Fine-tune with SFTTrainer

`assistant_only_loss=True` in `SFTConfig` masks the prompt tokens so the loss is computed only on the assistant's `<tool_call>` response — more efficient than full-sequence training and avoids accidentally reinforcing the system prompt wording.

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

dataset = Dataset.from_list([{"messages": r["messages"]} for r in correct])

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")

sft_config = SFTConfig(
    output_dir="reasoning-gym-chain-sum-Qwen3-1.7B-sft",
    max_length=MAX_SEQ_LEN,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()
trainer.push_to_hub(commit_message="SFT warm-up on reasoning_gym chain_sum")

---

## 10. Evaluate: before vs after

Run both the base model and the SFT checkpoint on a held-out set with a different seed.
The key metric is **format compliance** — how often the model uses `<tool_call>` correctly.

In [ ]:
import re
from transformers import pipeline
from reasoning_gym_env.client import ReasoningGymEnv
from reasoning_gym_env.models import ReasoningGymAction

async def evaluate_model(model_name, n_eval=50, seed=999):
    gen = pipeline(
        "text-generation",
        model=model_name,
        tokenizer=model_name,
        device_map="auto",
        dtype="auto",
    )
    gen.model.generation_config.max_length = None
    tok = AutoTokenizer.from_pretrained(model_name)
    env = ReasoningGymEnv(base_url="https://sergiopaniego-reasoning-gym.hf.space")

    obs = await env.reset(
        dataset_name="chain_sum",
        dataset_config=DATASET_CONFIG,
        seed=seed,
        size=n_eval,
    )

    rewards, format_hits = [], 0

    for i in range(n_eval):
        if i > 0:
            obs = await env.reset()

        question = obs.observation.question
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
        prompt = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        completion = gen(prompt, max_new_tokens=64)[0]["generated_text"][len(prompt):]

        m = re.search(r'"answer"\s*:\s*"?(\d+)"?', completion)
        if m:
            format_hits += 1
            answer = m.group(1)
        else:
            nums = re.findall(r"\b(\d+)\b", completion)
            answer = nums[-1] if nums else "0"

        result = await env.step(ReasoningGymAction(answer=answer))
        rewards.append(float(result.observation.score or 0.0))

    await env.close()
    del gen

    return {
        "accuracy": sum(rewards) / len(rewards),
        "format_compliance": format_hits / n_eval,
    }

In [ ]:
base_metrics = await evaluate_model("Qwen/Qwen3-1.7B")
sft_metrics = await evaluate_model(f"{YOUR_HF_USERNAME}/reasoning-gym-chain-sum-Qwen3-1.7B-sft")

print(f"\n{'Metric':<25} {'Base model':>12} {'After SFT':>12} {'Delta':>10}")
print("-" * 62)
for key, label in [("format_compliance", "Format compliance"), ("accuracy", "Accuracy")]:
    b, s = base_metrics[key], sft_metrics[key]
    print(f"{label:<25} {b:>12.1%} {s:>12.1%} {(s - b) * 100:>+9.1f} pp")

---

## 11. Where to go next: GRPO

The SFT checkpoint is ready to use as the warm-start for GRPO. In the
[end-to-end walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough),
change one line in the trainer setup:

```python
# Before (cold-start from the base model):
MODEL_NAME = "Qwen/Qwen3-1.7B"

# After (warm-start from your SFT checkpoint):
MODEL_NAME = f"{YOUR_HF_USERNAME}/reasoning-gym-chain-sum-Qwen3-1.7B-sft"
```

With format compliance already near 100%, GRPO's `reward_std` will be non-zero from the first batch
and the reward curve will climb immediately.